# Normal peak RPKM calculation

This notebook calculates peak-level RPKM from the 12 normal-peak overlap files. It keeps the overlap count, calculates each peak's own length, and uses the total number of records in the matching real or simulated damage BED file. This calculation is for the get rpkm boxplots across diffrent heterovhromatin states.

In [11]:
from pathlib import Path

import numpy as np
import pandas as pd

In [12]:
BASE_DIR = Path("/cta/users/guneyn23")
OVERLAP_DIR = BASE_DIR / "normal_peak_overlaps"
DAMAGE_DIR = BASE_DIR / "damageseq_data"
OUTPUT_DIR = BASE_DIR / "rpkm/normal_peaks"

CPD_REAL = DAMAGE_DIR / "R3Hela_1mCPD_GATCAG_S6_hg38_primary_assembly_DS.bed"
CPD_SIM = DAMAGE_DIR / "simulation/R3Hela_1mCPD_GATCAG_S6_hg38_primary_assembly_DS_sim.bed"
PP64_REAL = DAMAGE_DIR / "R3Hela_1m64_CGATGT_S5_hg38_primary_assembly_DS.bed"
PP64_SIM = DAMAGE_DIR / "simulation/R3Hela_1m64_CGATGT_S5_hg38_primary_assembly_DS_sim.bed"

In [13]:
JOBS = [
    ("ATAC_CPD_real", CPD_REAL),
    ("ATAC_CPD_simulated", CPD_SIM),
    ("ATAC_64_real", PP64_REAL),
    ("ATAC_64_simulated", PP64_SIM),
    ("H3K27me3_CPD_real", CPD_REAL),
    ("H3K27me3_CPD_simulated", CPD_SIM),
    ("H3K27me3_64_real", PP64_REAL),
    ("H3K27me3_64_simulated", PP64_SIM),
    ("H3K9me3_CPD_real", CPD_REAL),
    ("H3K9me3_CPD_simulated", CPD_SIM),
    ("H3K9me3_64_real", PP64_REAL),
    ("H3K9me3_64_simulated", PP64_SIM),
]

for name, damage_file in JOBS:
    overlap_file = OVERLAP_DIR / f"{name}_overlap.bed"
    if not overlap_file.is_file() or overlap_file.stat().st_size == 0:
        raise FileNotFoundError(f"Missing or empty overlap file: {overlap_file}")
    if not damage_file.is_file() or damage_file.stat().st_size == 0:
        raise FileNotFoundError(f"Missing or empty damage file: {damage_file}")

print("All 12 overlap files and all damage files are ready.")

All 12 overlap files and all damage files are ready.


## RPKM formula

For every normal peak:

$$RPKM = \frac{overlap\ count \times 10^9}{total\ damage\ records \times peak\ length}$$

Unlike the earlier 50-bp window calculation, `peak_length = end - start` is calculated separately for each normal peak.

In [14]:
PEAK_COLUMNS = [
    "chrom", "start", "end", "name", "score",
    "strand", "signal_value", "p_value", "q_value", "peak",
]

def count_nonempty_lines(file_path):
    with file_path.open() as infile:
        return sum(1 for line in infile if line.strip())

def calculate_peak_rpkm(overlap_file, total_damage_records):
    data = pd.read_csv(overlap_file, sep="\t", header=None)

    if data.shape[1] != 11:
        raise ValueError(
            f"{overlap_file}: expected 11 columns, found {data.shape[1]}"
        )

    data.columns = PEAK_COLUMNS + ["damage_count"]
    data["peak_length"] = data["end"] - data["start"]

    if (data["peak_length"] <= 0).any():
        raise ValueError(f"{overlap_file}: found a peak with non-positive length")
    if (data["damage_count"] < 0).any():
        raise ValueError(f"{overlap_file}: found a negative damage count")

    data["total_damage_records"] = total_damage_records
    data["rpkm"] = (
        data["damage_count"] * 1_000_000_000
        / (total_damage_records * data["peak_length"])
    )
    return data

In [15]:
# Count each distinct damage file only once, then reuse the totals.
damage_totals = {
    damage_file: count_nonempty_lines(damage_file)
    for damage_file in {damage_file for _, damage_file in JOBS}
}

pd.DataFrame(
    [(path.name, total) for path, total in damage_totals.items()],
    columns=["damage_file", "total_damage_records"],
).sort_values("damage_file").reset_index(drop=True)

,damage_file,total_damage_records
0,R3Hela_1m64_CGATGT_S5_hg38_primary_assembly_DS...,24841279
1,R3Hela_1m64_CGATGT_S5_hg38_primary_assembly_DS...,24841151
2,R3Hela_1mCPD_GATCAG_S6_hg38_primary_assembly_D...,32539066
3,R3Hela_1mCPD_GATCAG_S6_hg38_primary_assembly_D...,32538961


In [16]:
results = {}

for name, damage_file in JOBS:
    overlap_file = OVERLAP_DIR / f"{name}_overlap.bed"
    results[name] = calculate_peak_rpkm(
        overlap_file,
        damage_totals[damage_file],
    )

summary = pd.DataFrame([
    {
        "dataset": name,
        "peaks": len(data),
        "total_overlaps": int(data["damage_count"].sum()),
        "zero_overlap_peaks": int((data["damage_count"] == 0).sum()),
        "mean_peak_length": data["peak_length"].mean(),
        "mean_rpkm": data["rpkm"].mean(),
    }
    for name, data in results.items()
])

summary

,dataset,peaks,total_overlaps,zero_overlap_peaks,mean_peak_length,mean_rpkm,median_rpkm
0,ATAC_CPD_real,48911,237909,3318,635.276257,0.246533,0.200602
1,ATAC_CPD_simulated,48911,268013,1236,635.276257,0.272102,0.256103
2,ATAC_64_real,48911,256280,2430,635.276257,0.339468,0.298189
3,ATAC_64_simulated,48911,236127,1749,635.276257,0.312778,0.294375
4,H3K27me3_CPD_real,103751,631652,6724,659.935017,0.282815,0.241353
5,H3K27me3_CPD_simulated,103751,665614,2681,659.935017,0.299825,0.285883
6,H3K27me3_64_real,103751,614986,5783,659.935017,0.356368,0.322045
7,H3K27me3_64_simulated,103751,573942,3840,659.935017,0.337709,0.322907
8,H3K9me3_CPD_real,15881,51460,2555,361.191298,0.275793,0.214911
9,H3K9me3_CPD_simulated,15881,52445,1422,361.191298,0.283620,0.260444


## Inspect a result before saving

The original overlap columns are preserved. The notebook adds `peak_length`, `total_damage_records`, and `rpkm`.

In [17]:
results["ATAC_CPD_real"].head(10)

,chrom,start,end,name,score,strand,signal_value,p_value,q_value,peak,damage_count,peak_length,total_damage_records,rpkm
0,chr18,24075920,24076713,.,1000,.,-1,662.367,-1,470,7,793,32539066,0.271281
1,chr7,134646517,134647350,.,1000,.,-1,611.325,-1,212,12,833,32539066,0.442722
2,chr1,184754562,184755619,.,1000,.,-1,591.872,-1,461,6,1057,32539066,0.174450
3,chr7,158818325,158819188,.,1000,.,-1,583.585,-1,361,15,863,32539066,0.534165
4,chr15,92831571,92832166,.,1000,.,-1,558.620,-1,361,6,595,32539066,0.309905
5,chr10,32980422,32981253,.,1000,.,-1,538.734,-1,294,12,831,32539066,0.443788
6,chr11,83071372,83072343,.,1000,.,-1,528.347,-1,549,7,971,32539066,0.221551
7,chr7,129958389,129959095,.,1000,.,-1,514.564,-1,374,6,706,32539066,0.261181
8,chr1,151281184,151282411,.,1000,.,-1,501.116,-1,922,7,1227,32539066,0.175327
9,chr1,152047699,152048755,.,1000,.,-1,491.722,-1,490,6,1056,32539066,0.174615


In [18]:
# Confirm that real and simulated files contain the same peaks in the same order.
for region in ("ATAC", "H3K27me3", "H3K9me3"):
    for damage in ("CPD", "64"):
        real = results[f"{region}_{damage}_real"]
        simulated = results[f"{region}_{damage}_simulated"]
        same_coordinates = np.array_equal(
            real[["chrom", "start", "end"]].to_numpy(),
            simulated[["chrom", "start", "end"]].to_numpy(),
        )
        if not same_coordinates:
            raise ValueError(f"Peak mismatch: {region} {damage} real vs simulated")

print("All real/simulated peak coordinates match.")

All real/simulated peak coordinates match.


## Save results

Run this cell only after checking the summary and preview above. Change `SAVE_OUTPUTS` to `True` to write the 12 TSV files and one summary TSV.

In [19]:
SAVE_OUTPUTS = True

if SAVE_OUTPUTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    for name, data in results.items():
        output_file = OUTPUT_DIR / f"{name}_rpkm.tsv"
        data.to_csv(output_file, sep="\t", index=False)
        print(f"Written: {output_file}")

    summary_file = OUTPUT_DIR / "normal_peak_rpkm_summary.tsv"
    summary.to_csv(summary_file, sep="\t", index=False)
    print(f"Written: {summary_file}")
else:
    print("Nothing was saved. Check the results, then set SAVE_OUTPUTS = True.")

Written: /cta/users/guneyn23/rpkm/normal_peaks/ATAC_CPD_real_rpkm.tsv
Written: /cta/users/guneyn23/rpkm/normal_peaks/ATAC_CPD_simulated_rpkm.tsv
Written: /cta/users/guneyn23/rpkm/normal_peaks/ATAC_64_real_rpkm.tsv
Written: /cta/users/guneyn23/rpkm/normal_peaks/ATAC_64_simulated_rpkm.tsv
Written: /cta/users/guneyn23/rpkm/normal_peaks/H3K27me3_CPD_real_rpkm.tsv
Written: /cta/users/guneyn23/rpkm/normal_peaks/H3K27me3_CPD_simulated_rpkm.tsv
Written: /cta/users/guneyn23/rpkm/normal_peaks/H3K27me3_64_real_rpkm.tsv
Written: /cta/users/guneyn23/rpkm/normal_peaks/H3K27me3_64_simulated_rpkm.tsv
Written: /cta/users/guneyn23/rpkm/normal_peaks/H3K9me3_CPD_real_rpkm.tsv
Written: /cta/users/guneyn23/rpkm/normal_peaks/H3K9me3_CPD_simulated_rpkm.tsv
Written: /cta/users/guneyn23/rpkm/normal_peaks/H3K9me3_64_real_rpkm.tsv
Written: /cta/users/guneyn23/rpkm/normal_peaks/H3K9me3_64_simulated_rpkm.tsv
Written: /cta/users/guneyn23/rpkm/normal_peaks/normal_peak_rpkm_summary.tsv
